In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser

In [21]:
import os

In [39]:
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?",
]

In [ ]:
## OpenAI Key
os.environ["OPENAI_API_KEY"] = ""

In [23]:
## Load Embeddings e LLM

embeddings_model = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo", max_tokens=200)

In [24]:
## Carregar PDF

pdf_link = "../os-sertoes.pdf"
loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

In [25]:
# Chunking

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)

In [26]:
# Salvar no Vector DB

db = Chroma.from_documents(
    chunks,
    embedding=embeddings_model,
    persist_directory="text_index"
)

In [27]:
# Carregar DB
vectordb = Chroma(persist_directory="text_index", embedding_function=embeddings_model)

In [28]:
# Load Retriever

retriever = db.as_retriever(search_kwargs={"k": 3})

In [42]:
# Prompt
TEMPLATE = """"
Você é um assistente de perguntas e respostas sobre o livro "Os Sertões" de Euclides da Cunha. Responda a pergunta abaixo utilizando o contexto informado.
Context: {context}
Pergunta: {question}
"""

prompt = PromptTemplate(input_variables=["context", "question"], template=TEMPLATE)
parser = StrOutputParser()
answer_chain = RunnableSequence(prompt | llm | parser)

In [43]:
def answer_question(question):
    context = retriever.invoke(question)
    response = answer_chain({"context": context, "question": question})
    return response

In [46]:
for index, question in enumerate(questions):
    result = rag_chain.invoke({"input": question})
    answer = result["answer"]
    print({"numero": index, "pergunta": question, "resposta": answer})

{'numero': 0, 'pergunta': 'Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?', 'resposta': 'Euclides da Cunha descreve o ambiente natural do sertão nordestino como uma região marcada pela aridez, pela falta de chuvas e pela vegetação de caatinga, que apresenta desafios e dificuldades para a vida dos habitantes locais. Ele destaca como a dureza desse ambiente influenciou a formação e a resistência do povo sertanejo, moldando sua cultura e sua maneira de lidar com as adversidades. A terra árida e as condições climáticas extremas do sertão foram determinantes para a construção da identidade e das estratégias de sobrevivência da população que ali vivia, refletindo-se em aspectos como a economia, a organização social e as lutas pela terra.'}
{'numero': 1, 'pergunta': 'Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambien